# Critical Input DEQN: Discretion Explicit I_A Light

Light global screening notebook for the explicit-`I_A` optimal-policy system. It intentionally does **not** use the active-state reference mode. The goal is to test whether repair investment appears endogenously in the broad/global training domain after making `I_A` a direct choice.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

try:
    import torch
    import pandas as pd
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy', 'scipy', 'pandas', 'matplotlib', 'tqdm'], check=True)
    import torch
    import pandas as pd

ROOT = Path('/content/econml') if Path('/content').exists() else Path.cwd()
REPO_URL = 'https://github.com/codist-posist/econml.git'
BRANCH = 'critical-input-deqn-baseline'

if not (ROOT / 'src' / 'critical_input_deqn').exists():
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=ROOT, check=True)

os.chdir(ROOT)
print('ROOT =', ROOT)
print('HEAD =', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT, text=True).strip())

In [ ]:
def run_stream(cmd, *, cwd=ROOT, env=None):
    print('Running:')
    print(' '.join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

In [ ]:
ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
OUT = ARTIFACT_ROOT / 'discretion_explicit_ia_light'
OUT.mkdir(parents=True, exist_ok=True)

KIND = 'discretion'

# Light global screening setup: explicit I_A is active in the code, but reference-state training flags are intentionally OFF.
STEPS = 3000
LR = 1e-4
DTYPE = 'float32'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

QMC_TRAIN = 128
QMC_VAL = 128
N_VAL_STATES = 512
STOP_VAL_STATES = 256

HIDDEN_WIDTH = 128
HIDDEN_DEPTH = 2
BATCH_SIZE = 1024
FULL_BATCH_SIZE = 128
SIM_BATCH_SIZE = 256
EPISODE_LENGTH = 16
EPISODE_UPDATES_PER_EPISODE = 1
EPISODE_BROAD_SHARE = 0.50

FEASIBILITY_PRETRAIN_STEPS = 400
FULL_WEIGHT_WARMUP_STEPS = 600

LOG_EVERY = 100
CHECKPOINT_EVERY = 500
CHECKPOINT_KEEP = 8

PRIVATE_LOSS_WEIGHT = 1.0
BELLMAN_LOSS_WEIGHT = 0.0
STATIONARITY_LOSS_WEIGHT = 1.0
ENVELOPE_LOSS_WEIGHT = 1.0
PROMISE_LOSS_WEIGHT = 1.0

SCENARIO_Q_WEIGHT = 10.0
CALM_ANCHOR_WEIGHT = 2.0
CALM_RESIDUAL_WEIGHT = 2.0
SCENARIO_BURNIN = 5
SCENARIO_HORIZON = 10
SCENARIO_LOSS_INTERVAL = 50
TARGET_SCENARIO_Q_RMS = 1e-2

Q_NOBUBBLE_WEIGHT = 0.25
Q_NOBUBBLE_HORIZON = 8
Q_NOBUBBLE_PATHS = 32
BEST_SCENARIO_Q_WEIGHT = 1.0
BEST_Q_NOBUBBLE_WEIGHT = 0.5
BEST_CALM_ANCHOR_WEIGHT = 0.5
BEST_CALM_RESIDUAL_WEIGHT = 0.5

PROMISE_INIT_SCALE = 1.0

print('OUT =', OUT)
print('DEVICE =', DEVICE, 'DTYPE =', DTYPE)
print('This notebook uses the global explicit-I_A training command.')

In [ ]:
cmd = [
    sys.executable, '-u', '-m', 'src.critical_input_deqn.run_optimal',
    '--output-dir', str(OUT),
    '--kind', KIND,
    '--steps', str(STEPS),
    '--lr', str(LR),
    '--qmc-train', str(QMC_TRAIN),
    '--qmc-val', str(QMC_VAL),
    '--n-val-states', str(N_VAL_STATES),
    '--hidden-width', str(HIDDEN_WIDTH),
    '--hidden-depth', str(HIDDEN_DEPTH),
    '--device', DEVICE,
    '--dtype', DTYPE,
    '--stop-val-states', str(STOP_VAL_STATES),
    '--log-every', str(LOG_EVERY),
    '--batch-size', str(BATCH_SIZE),
    '--full-batch-size', str(FULL_BATCH_SIZE),
    '--sim-batch-size', str(SIM_BATCH_SIZE),
    '--episode-length', str(EPISODE_LENGTH),
    '--episode-updates-per-episode', str(EPISODE_UPDATES_PER_EPISODE),
    '--episode-broad-share', str(EPISODE_BROAD_SHARE),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--checkpoint-keep', str(CHECKPOINT_KEEP),
    '--scenario-q-weight', str(SCENARIO_Q_WEIGHT),
    '--calm-anchor-weight', str(CALM_ANCHOR_WEIGHT),
    '--calm-residual-weight', str(CALM_RESIDUAL_WEIGHT),
    '--scenario-burnin', str(SCENARIO_BURNIN),
    '--scenario-horizon', str(SCENARIO_HORIZON),
    '--scenario-loss-interval', str(SCENARIO_LOSS_INTERVAL),
    '--target-scenario-q-rms', str(TARGET_SCENARIO_Q_RMS),
    '--feasibility-pretrain-steps', str(FEASIBILITY_PRETRAIN_STEPS),
    '--full-weight-warmup-steps', str(FULL_WEIGHT_WARMUP_STEPS),
    '--private-loss-weight', str(PRIVATE_LOSS_WEIGHT),
    '--bellman-loss-weight', str(BELLMAN_LOSS_WEIGHT),
    '--stationarity-loss-weight', str(STATIONARITY_LOSS_WEIGHT),
    '--envelope-loss-weight', str(ENVELOPE_LOSS_WEIGHT),
    '--promise-loss-weight', str(PROMISE_LOSS_WEIGHT),
    '--q-nobubble-weight', str(Q_NOBUBBLE_WEIGHT),
    '--q-nobubble-horizon', str(Q_NOBUBBLE_HORIZON),
    '--q-nobubble-paths', str(Q_NOBUBBLE_PATHS),
    '--best-scenario-q-weight', str(BEST_SCENARIO_Q_WEIGHT),
    '--best-q-nobubble-weight', str(BEST_Q_NOBUBBLE_WEIGHT),
    '--best-calm-anchor-weight', str(BEST_CALM_ANCHOR_WEIGHT),
    '--best-calm-residual-weight', str(BEST_CALM_RESIDUAL_WEIGHT),
]
if KIND == 'commitment':
    cmd += ['--promise-init-scale', str(PROMISE_INIT_SCALE)]
assert not any(('active' in str(x)) and ('reference' in str(x)) for x in cmd)
run_stream(cmd, cwd=ROOT)

In [ ]:
eval_path = OUT / f'{KIND}_eval.json'
with eval_path.open('r', encoding='utf-8') as fh:
    metrics = json.load(fh)

wanted = [
    'rms', 'max_abs',
    'priv_resource.rms', 'priv_price_index.rms', 'priv_calvo_S.rms', 'priv_calvo_F.rms', 'priv_Q.rms', 'priv_repair_KKT.rms',
    'stat_C.rms', 'stat_Y.rms', 'stat_Pi.rms', 'stat_Q_A.rms', 'stat_I_A.rms', 'stat_S_p.rms', 'stat_F_p.rms',
    'env_A.rms', 'env_log_Delta.rms',
    'promise_S.rms', 'promise_F.rms', 'promise_Q.rms',
    'scenario_Q.rms', 'scenario_Q_pv.rms',
    'scenario_repair_activation.max', 'scenario_I_A.max', 'scenario_cap_pressure.max',
    'exact_repair_gap_scaled.rms', 'exact_repair_projection.rms', 'exact_euler_rate_residual.rms',
    'head_saturation.freq',
]
summary = pd.DataFrame([{'metric': k, 'value': metrics.get(k)} for k in wanted if k in metrics])
display(summary)

In [ ]:
# Quick checkpoint scan with the current code. This is intentionally light.
import torch
from src.critical_input_deqn.config import BaselineParams, NetworkConfig, QMCConfig, TrainConfig
from src.critical_input_deqn.train import (
    evaluate_optimal,
    load_model_state_dict,
    make_commitment_net,
    make_discretion_net,
)

CHECKPOINT_SCAN = True
SCAN_QMC = 64
SCAN_STATES = 128

def checkpoint_candidates(out_dir, kind):
    ckpt_dir = out_dir / 'checkpoints'
    items = []
    best = ckpt_dir / f'{kind}_best.pt'
    if best.exists():
        items.append(('best', best))
    for path in sorted(ckpt_dir.glob(f'{kind}_step_*.pt')):
        items.append((path.stem.replace(f'{kind}_', ''), path))
    final = out_dir / f'{kind}.pt'
    if final.exists():
        items.append(('final', final))
    seen = set()
    unique = []
    for label, path in items:
        if path not in seen:
            unique.append((label, path))
            seen.add(path)
    return unique

if CHECKPOINT_SCAN:
    dtype_t = torch.float32 if DTYPE == 'float32' else torch.float64
    params = BaselineParams()
    net_cfg = NetworkConfig(hidden_width=HIDDEN_WIDTH, hidden_depth=HIDDEN_DEPTH)
    qmc_cfg = QMCConfig(n_train=SCAN_QMC, n_val=SCAN_QMC, seed=987)
    scan_cfg = TrainConfig(device=DEVICE, dtype=dtype_t, show_progress=False, optimal_full_batch_size=64)
    rows = []
    for label, path in checkpoint_candidates(OUT, KIND):
        net = make_discretion_net(net_cfg, device=DEVICE, dtype=dtype_t) if KIND == 'discretion' else make_commitment_net(net_cfg, device=DEVICE, dtype=dtype_t)
        payload = torch.load(path, map_location=DEVICE)
        state = payload.get('state_dict', payload)
        load_model_state_dict(net, state)
        m = evaluate_optimal(net, kind=KIND, params=params, qmc_cfg=qmc_cfg, train_cfg=scan_cfg, n_states=SCAN_STATES)
        rows.append({
            'checkpoint_label': label,
            'path': str(path),
            'rms': m.get('rms'),
            'max_abs': m.get('max_abs'),
            'priv_repair_KKT.rms': m.get('priv_repair_KKT.rms'),
            'stat_I_A.rms': m.get('stat_I_A.rms'),
            'scenario_Q.rms': m.get('scenario_Q.rms'),
            'scenario_Q_pv.rms': m.get('scenario_Q_pv.rms'),
            'scenario_repair_activation.max': m.get('scenario_repair_activation.max'),
            'scenario_I_A.max': m.get('scenario_I_A.max'),
            'scenario_cap_pressure.max': m.get('scenario_cap_pressure.max'),
            'head_saturation.freq': m.get('head_saturation.freq'),
        })
    scan = pd.DataFrame(rows)
    display(scan)
    scan.to_csv(OUT / f'{KIND}_light_checkpoint_scan.csv', index=False)

In [ ]:
# Optional economic IR diagnostics. Keep this off during quick screening; turn on after the run looks useful.
RUN_IR_DIAGNOSTICS = False
if RUN_IR_DIAGNOSTICS:
    from src.critical_input_deqn.notebook_diagnostics import optimal_ir_mechanism_diagnostics
    ir_labels, ir_defs, mechanism_table = optimal_ir_mechanism_diagnostics(
        artifact_root=ARTIFACT_ROOT,
        output_dir=OUT,
        kind=KIND,
        device=DEVICE,
        dtype=DTYPE,
        hidden_width=HIDDEN_WIDTH,
        hidden_depth=HIDDEN_DEPTH,
    )
    display(mechanism_table)

In [ ]:
zip_base = Path('/content') / OUT.name if Path('/content').exists() else OUT
zip_path = shutil.make_archive(str(zip_base), 'zip', OUT)
print('Saved zip:', zip_path)
print('Size MB:', Path(zip_path).stat().st_size / 1024**2)